# Train a small AdjMatSeer on OpenBabel-labelled bonds

Data comes from `generate_obabel_pairs.ipynb` (teacher-pair x1 or 420 EDM @ 100 steps — not FM).
The CLI twin is `train_small_seer.py`; this notebook is the preferred entry point.

Training logic follows `./conformer_train.py`, the procedure behind the production 2048 seer:
full-matrix cross-entropy (`matrix_criteria`) plus the `wrong_bonds` metric. Three additions:

* **Canonical atom order.** The seer is order-sensitive: at inference `prepare_adj_mat_seer_input`
  runs `canonicalise` (RDKit DetermineConnectivity, then renumber by `_smilesAtomOutputOrder`)
  before building inputs. Shards must be in that same order or training and inference disagree.
  `CANONICAL = "check"` audits a sample, `"apply"` re-derives the order from the stored geometry
  and permutes elements / coords / conn / target together. Verified: re-canonicalisation through
  the xyz path is idempotent except for highly symmetric molecules (canonical-rank ties, e.g.
  crown ethers), where inference has the same ambiguity — so a few percent "reordered" is normal.
* **Warm start.** Slices a wider trained seer down to `N_HIDDEN` by magnitude-ranked channel
  selection, the same trick that rescued EGNN width distillation. Probing the sliced init against
  the trained 2048 seer on real inference inputs: ~97% argmax agreement with the teacher at
  `HEAD_SCALE = 1.0` (vs ~13% cold), no logit explosion.
* **Validity-based selection:** RDKit sanitization of the reconstructed graph is the number the
  README quotes as 48% (ML) vs 93% (OpenBabel).

Shard schema (`./bond_pairs/shard*.pt`):

| key | dtype / shape | meaning |
|---|---|---|
| `elements` | int8, DIMENSION | atomic numbers, canonical order |
| `coords` | f16, DIMENSION x 3 | canonical order, Angstrom |
| `conn` | int8, D x D | RDKit DetermineConnectivity guess -> model INPUT |
| `target` | int8, D x D | OpenBabel bond orders 0-4 -> model TARGET |
| `n_atoms` | int16 | real atom count |

Inputs are rebuilt to match `prepare_adj_mat_seer_input` exactly: dist_mat + I, and binary
(conn + I) clamped to 1. Nothing here may use `target` to build an input.

Sizes: 2048-wide baseline ~21.8 M params / 83 MB; 512 -> ~1.6 M / 6 MB, 256 -> ~0.5 M / 2 MB.
Seer compute is only ~5% of an 8-NFE sample, so expect a packaging win, not a speed win.

No DataParallel: `GraphConv.l_norm` pins its output to a fixed `self.device`, so replicas would
all write to one GPU. Use a single device.

In [ ]:
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

REPO = Path.cwd().resolve()
while not (REPO / "src" / "mlconfgen").exists():
    if REPO.parent == REPO:
        raise FileNotFoundError("repo root with src/mlconfgen not found above cwd")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from rdkit import Chem, RDLogger
from rdkit.Chem import rdDetermineBonds

from src.mlconfgen.adj_mat_seer import AdjMatSeer
from src.mlconfgen.utils.common import bond_type_dict, elements_dict
from src.mlconfgen.utils.config import DIMENSION, NUM_BOND_TYPES

RDLogger.DisableLog("rdApp.*")
print("repo:", REPO)

In [ ]:
# ------------------------------- config: edit and rerun -------------------------------
N_HIDDEN = 512
EPOCHS = 30
BATCH = 128
LR = 3e-4                # conformer_train.py used 5e-4 cold; keep lower for a warm start
VAL_FRAC = 0.02
MAX_SHARDS = None        # int to cap shards while smoke-testing

LOSS = "matrix"          # "matrix" = full-matrix CE from conformer_train.py (production
                         # recipe) | "masked" = class-weighted CE on real upper-tri pairs
CANONICAL = "check"      # "off" | "check" (audit a sample) | "apply" (permute everything)
CANONICAL_SAMPLE = 2000  # molecules to audit when CANONICAL == "check"

WARM_START = REPO / "adj_mat_seer_chembl_15_39.pt"  # None for a cold start
HEAD_SCALE = 1.0         # shrink the sliced `resize` logit head; 1.0 keeps the teacher head

WEIGHT_CAP = 25.0        # cap on inverse-frequency class weight (masked loss only)
N_VALIDITY = 2000        # molecules per eval for the validity metric
BASELINE = REPO / "adj_mat_seer_chembl_15_39.pt"    # None to skip baseline scoring
SELECT = "valid"         # checkpoint selection: "valid" | "wrong"
PATIENCE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

DATA_DIR = Path("./bond_pairs")
OUT_DIR = Path("./checkpoints_seer")

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / f"train_seer_{N_HIDDEN}.log"


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


torch.manual_seed(SEED)
print(f"device={DEVICE}  n_hidden={N_HIDDEN}  loss={LOSS}")

In [ ]:
# ------------------------------------ data + canonical order ------------------------------------
class BondData:
    """All shards in RAM (int8/f16 -> ~3.6 kB per molecule)."""

    def __init__(self, data_dir: Path, max_shards, log):
        shards = sorted(data_dir.glob("shard*.pt"))
        if not shards:
            raise FileNotFoundError(f"no shard*.pt in {data_dir}")
        if max_shards:
            shards = shards[:max_shards]
        packs = [torch.load(p, map_location="cpu", weights_only=False) for p in shards]
        self.elements = torch.cat([p["elements"] for p in packs])
        self.coords = torch.cat([p["coords"] for p in packs])
        self.conn = torch.cat([p["conn"] for p in packs])
        self.target = torch.cat([p["target"] for p in packs])
        self.n_atoms = torch.cat([p["n_atoms"] for p in packs])
        self.aromatic_mode = packs[0].get("aromatic_mode", "?")
        nbytes = sum(t.numel() * t.element_size() for t in
                     (self.elements, self.coords, self.conn, self.target))
        log(f"loaded {len(shards)} shards  n={self.n_atoms.shape[0]} "
            f"arom={self.aromatic_mode}  ram={nbytes / 1e9:.2f} GB")

    def __len__(self):
        return self.n_atoms.shape[0]


def build_inputs(d: BondData, idx: torch.Tensor, device: str):
    """-> elements(long), dist_mat(float), adj_in(float), target(long), n_atoms(long)

    Mirrors `prepare_adj_mat_seer_input`: distance matrix + I, and the binary
    connectivity guess + I. Assumes rows are already in canonical atom order (see
    CANONICAL); this function must not reorder anything, or the padded block and
    the stored target would drift apart.
    """
    elements = d.elements[idx].to(device=device, dtype=torch.long)
    coords = d.coords[idx].to(device=device, dtype=torch.float32)
    conn = d.conn[idx].to(device=device, dtype=torch.float32)
    target = d.target[idx].to(device=device, dtype=torch.long)
    n_atoms = d.n_atoms[idx].to(device=device, dtype=torch.long)

    eye = torch.eye(DIMENSION, device=device)
    # padded rows are zero in coords, so zero them in the distance matrix too
    valid = (torch.arange(DIMENSION, device=device)[None, :] < n_atoms[:, None]).float()
    pair_valid = valid[:, :, None] * valid[:, None, :]
    dist = torch.cdist(coords, coords) * pair_valid + eye
    adj_in = (conn + eye).clamp(max=1.0)
    return elements, dist, adj_in, target, n_atoms


def pair_mask(n_atoms: torch.Tensor, device: str) -> torch.Tensor:
    """Upper triangle of the real n x n block; the model output is symmetric."""
    ar = torch.arange(DIMENSION, device=device)
    valid = ar[None, :] < n_atoms[:, None]
    both = valid[:, :, None] & valid[:, None, :]
    return both & (ar[None, :, None] < ar[None, None, :])


def canonical_order(elements: np.ndarray, coords: np.ndarray, n: int):
    """The permutation `mlconfgen`'s `canonicalise` would apply to this geometry.

    Rebuilds the xyz the inference path sees, runs RDKit DetermineConnectivity and reads
    `_smilesAtomOutputOrder`. Returns `order` where new atom i is old atom order[i] (what
    `Chem.RenumberAtoms` takes), or None if RDKit cannot read the geometry.
    """
    lines = [str(n), ""]
    for i in range(n):
        z = int(elements[i])
        if z not in elements_dict:
            return None
        lines.append(
            f"{elements_dict[z]} {coords[i, 0]:.9f} {coords[i, 1]:.9f} {coords[i, 2]:.9f}"
        )
    mol = Chem.MolFromXYZBlock("\n".join(lines) + "\n")
    if mol is None or mol.GetNumAtoms() != n:
        return None
    try:
        rdDetermineBonds.DetermineConnectivity(mol)
        Chem.MolToSmiles(mol)
        raw = mol.GetProp("_smilesAtomOutputOrder")
    except Exception:
        return None
    order = [int(x) for x in raw.replace("[", "").replace("]", "").split(",") if x != ""]
    if len(order) != n:
        return None
    return np.asarray(order, dtype=np.int64)


def enforce_canonical(d: BondData, mode: str, sample: int, log) -> None:
    """`check` audits a sample; `apply` permutes every molecule into canonical order."""
    if mode == "off":
        log("canonical: skipped (CANONICAL='off') — only safe if shards were written "
            "by generate_obabel_pairs.ipynb, which canonicalises before labelling")
        return

    n_total = len(d)
    idxs = range(n_total) if mode == "apply" else range(min(sample, n_total))
    els_np = d.elements.numpy()
    crd_np = d.coords.float().numpy()
    same = moved = failed = 0

    for i in tqdm(idxs, desc=f"canonical:{mode}"):
        n = int(d.n_atoms[i])
        order = canonical_order(els_np[i], crd_np[i], n)
        if order is None:
            failed += 1
            continue
        if np.array_equal(order, np.arange(n)):
            same += 1
            continue
        moved += 1
        if mode != "apply":
            continue
        full = torch.arange(DIMENSION)
        full[:n] = torch.from_numpy(order)  # padding maps to itself
        d.elements[i] = d.elements[i][full]
        d.coords[i] = d.coords[i][full]
        d.conn[i] = d.conn[i][full][:, full]
        d.target[i] = d.target[i][full][:, full]

    checked = same + moved + failed
    log(f"canonical:{mode}  checked={checked}  already_canonical={same} "
        f"reordered={moved}  rdkit_failed={failed}")
    # Highly symmetric molecules (e.g. crown ethers) have canonical-rank ties that RDKit
    # breaks by input order, so a few percent 'reordered' is normal even on canonical
    # data; inference has the same ambiguity for them. A large fraction means the shards
    # were never canonicalised.
    if mode == "check" and checked and moved / checked > 0.05:
        log(f"WARN {moved}/{checked} molecules are NOT in inference canonical order — "
            f"rerun with CANONICAL='apply', or training will not match inference")


d = BondData(DATA_DIR, MAX_SHARDS, log)
enforce_canonical(d, CANONICAL, CANONICAL_SAMPLE, log)

perm = torch.randperm(len(d), generator=torch.Generator().manual_seed(SEED))
n_val = max(1, int(len(d) * VAL_FRAC))
val_idx, train_idx = perm[:n_val], perm[n_val:]
log(f"split train={train_idx.shape[0]} val={val_idx.shape[0]}")

In [ ]:
# ------------------------------------ loss + metrics ------------------------------------
def matrix_criteria(output: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Full-matrix cross-entropy over every cell, as in ./conformer_train.py.

    output (B, D, D, C) logits, target (B, D, D) class ids. The reference feeds one-hot
    targets to `F.cross_entropy`; for exact one-hot that is numerically the same as class
    ids. Both triangles and the padded block count (padding is class 0), which is what the
    production seer was trained on.
    """
    return F.cross_entropy(output.reshape(-1, output.shape[-1]), target.reshape(-1))


def class_weights(d: BondData, n_sample: int, cap: float, device: str) -> torch.Tensor:
    """Inverse sqrt-frequency, capped. ~94% of pairs are 'no bond'."""
    idx = torch.arange(min(n_sample, len(d)))
    tgt = d.target[idx].to(torch.long)
    na = d.n_atoms[idx].to(torch.long)
    m = pair_mask(na, "cpu")
    vals = tgt[m]
    counts = torch.bincount(vals, minlength=NUM_BOND_TYPES).float().clamp_min(1.0)
    w = (counts.sum() / counts).sqrt()
    w = (w / w[0]).clamp(max=cap)
    return w.to(device)


def compute_loss(output, target, n_atoms, mode: str, weights: torch.Tensor):
    if mode == "matrix":
        return matrix_criteria(output, target)
    m = pair_mask(n_atoms, output.device)
    return F.cross_entropy(output[m], target[m], weight=weights)


def wrong_bonds(output: torch.Tensor, target: torch.Tensor, n_atoms: torch.Tensor) -> dict:
    """Per-molecule fraction of mispredicted pairs over the real n x n upper triangle.

    `conformer_train.py` calls `wrong_bonds(output, target)["wrong"]`; that helper lives in
    the Structure Seer repo, not here, so this is the equivalent restricted to real atoms
    (padding excluded, and only one triangle since the model output is symmetric).
    """
    pred = output.argmax(-1)
    m = pair_mask(n_atoms, output.device)
    wrong = ((pred != target) & m).flatten(1).sum(1).float()
    total = m.flatten(1).sum(1).clamp_min(1).float()
    return {"wrong": wrong / total}


def mol_is_valid(elements: np.ndarray, orders: np.ndarray, n: int) -> bool:
    m = Chem.RWMol()
    for z in elements[:n]:
        m.AddAtom(Chem.Atom(int(z)))
    for i in range(n):
        for j in range(i):
            o = int(orders[i, j])
            if o:
                m.AddBond(j, i, bond_type_dict[o])
    try:
        Chem.SanitizeMol(m.GetMol())
        return True
    except Exception:
        return False


@torch.inference_mode()
def evaluate(model, d: BondData, idx: torch.Tensor, device: str, batch: int,
             weights: torch.Tensor, n_validity: int, loss_mode: str):
    """Loss, wrong-bond rate, bond P/R/F1, exact-graph match, molecule validity."""
    model.eval()
    loss_sum = n_batch = 0.0
    tp = fp = fn = cls_ok = cls_tot = 0
    exact = n_mol = 0
    valid = checked = 0
    wrong_all = []

    for s in range(0, idx.shape[0], batch):
        bidx = idx[s : s + batch]
        el, dist, adj_in, tgt, na = build_inputs(d, bidx, device)
        logits = model(elements=el, dist_mat=dist, adj_mat=adj_in)

        loss_sum += compute_loss(logits, tgt, na, loss_mode, weights).item()
        n_batch += 1
        wrong_all.extend(wrong_bonds(logits, tgt, na)["wrong"].tolist())

        m = pair_mask(na, device)
        lg, tg = logits[m], tgt[m]
        pred = lg.argmax(-1)
        pb, tb = pred > 0, tg > 0
        tp += (pb & tb).sum().item()
        fp += (pb & ~tb).sum().item()
        fn += (~pb & tb).sum().item()
        cls_ok += (pred[tb] == tg[tb]).sum().item()
        cls_tot += int(tb.sum().item())

        full = logits.argmax(-1)
        bad = ((full != tgt) & m).flatten(1).any(1)
        exact += int((~bad).sum().item())
        n_mol += bidx.shape[0]

        if checked < n_validity:
            take = min(bidx.shape[0], n_validity - checked)
            orders = full.cpu().numpy()
            els = d.elements[bidx[:take]].numpy()
            nas = d.n_atoms[bidx[:take]].numpy()
            for k in range(take):
                valid += mol_is_valid(els[k], orders[k], int(nas[k]))
            checked += take

    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return {
        "loss": loss_sum / max(n_batch, 1),
        "wrong": sum(wrong_all) / max(len(wrong_all), 1),
        "bond_p": prec,
        "bond_r": rec,
        "bond_f1": 2 * prec * rec / max(prec + rec, 1e-9),
        "order_acc": cls_ok / max(cls_tot, 1),
        "exact": exact / max(n_mol, 1),
        "valid": valid / max(checked, 1),
        "n_valid_checked": checked,
    }


w = class_weights(d, 20_000, WEIGHT_CAP, DEVICE)
log("class weights " + " ".join(f"{k}:{v:.2f}" for k, v in enumerate(w.tolist())))

In [ ]:
# --------------------------- model init: warm start + baseline ---------------------------
def _row_norm(W):
    return W.pow(2).sum(1)


def _col_norm(W):
    return W.pow(2).sum(0)


def _topk(score, k):
    return torch.topk(score, k).indices.sort().values


@torch.no_grad()
def warm_start_from(big: AdjMatSeer, small: AdjMatSeer, head_scale: float = 1.0) -> None:
    """Init a narrow AdjMatSeer by magnitude-ranked channel slicing of a wider trained one.

    Both branches are plain chains of `GraphConv` linears, so every hidden width is a
    private interface between one producer (rows of its Linear) and one consumer (columns
    of the next). Channels are ranked by producer row-norm + consumer col-norm and the top
    `n_hidden` kept. Embeddings, `nodes_coord_fc` and both output shapes are
    width-independent and copied verbatim.

    `head_scale` shrinks the `resize` logit head only. Unlike the EGNN case there is no
    residual stream compounding across blocks here, and probing the sliced init against
    the trained 2048 seer showed no logit explosion and ~97% argmax agreement with the
    teacher at head_scale=1.0 (vs ~13% for a cold init), so 1.0 is the default.
    `dm_resize` is deliberately never scaled — it is the geometry bottleneck feeding
    `nodes_coord_fc`, and damping it would throw away the distance-matrix signal.
    """
    hs = small.gcn1.linear.out_features

    def do_chain(chain_b, chain_s, head_b, head_s, scale):
        prev = None  # first input space is embedding_dim -> width-independent
        for k, (gb, gs) in enumerate(zip(chain_b, chain_s)):
            Wb, bb = gb.linear.weight, gb.linear.bias
            consumer = chain_b[k + 1].linear.weight if k + 1 < len(chain_b) else head_b.weight
            keep = _topk(_row_norm(Wb) + _col_norm(consumer), hs)
            W = Wb[keep]
            if prev is not None:
                W = W[:, prev]
            gs.linear.weight.data.copy_(W)
            gs.linear.bias.data.copy_(bb[keep])
            prev = keep
        head_s.weight.data.copy_(head_b.weight[:, prev] * scale)
        head_s.bias.data.copy_(head_b.bias * scale)

    do_chain([big.gcn1, big.gcn2, big.gcn3, big.gcn4],
             [small.gcn1, small.gcn2, small.gcn3, small.gcn4],
             big.resize, small.resize, head_scale)
    do_chain([big.gcn1_dm, big.gcn2_dm, big.gcn3_dm],
             [small.gcn1_dm, small.gcn2_dm, small.gcn3_dm],
             big.dm_resize, small.dm_resize, 1.0)

    small.nodes_embedding.weight.data.copy_(big.nodes_embedding.weight)
    small.dm_nodes_embedding.weight.data.copy_(big.dm_nodes_embedding.weight)
    small.nodes_coord_fc.weight.data.copy_(big.nodes_coord_fc.weight)
    small.nodes_coord_fc.bias.data.copy_(big.nodes_coord_fc.bias)


def load_seer(path: Path, device: str, n_hidden=None) -> AdjMatSeer:
    """Load a seer checkpoint, inferring n_hidden from the weights when not given."""
    sd = torch.load(path, map_location=device, weights_only=False)
    sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
    if n_hidden is None:
        n_hidden = sd["gcn1.linear.weight"].shape[0]
    model = AdjMatSeer(n_hidden=int(n_hidden), device=device).to(device)
    model.load_state_dict(sd)
    return model


model = AdjMatSeer(n_hidden=N_HIDDEN, device=DEVICE).to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
log(f"AdjMatSeer n_hidden={N_HIDDEN}  params={n_par / 1e6:.2f} M "
    f"({n_par * 4 / 1e6:.1f} MB fp32)  loss={LOSS}")

if WARM_START is not None:
    src = load_seer(Path(WARM_START), DEVICE)
    hs = src.gcn1.linear.out_features
    if hs <= N_HIDDEN:
        raise ValueError(f"warm-start width {hs} must exceed N_HIDDEN {N_HIDDEN}")
    warm_start_from(src, model, head_scale=HEAD_SCALE)
    log(f"warm-started {hs}->{N_HIDDEN} from {Path(WARM_START).name} "
        f"(keep {N_HIDDEN / hs:.0%} of channels, head_scale={HEAD_SCALE})")
    del src
    if str(DEVICE).startswith("cuda"):
        torch.cuda.empty_cache()

if BASELINE is not None and Path(BASELINE).exists():
    base = load_seer(Path(BASELINE), DEVICE, n_hidden=2048)
    bm = evaluate(base, d, val_idx, DEVICE, BATCH, w, N_VALIDITY, LOSS)
    log(f"BASELINE 2048  valid={bm['valid']:.3f}  <- the only comparable number "
        f"(README quotes 48% for this model); exact={bm['exact']:.3f} "
        f"wrong={bm['wrong']:.4f} bond_f1={bm['bond_f1']:.4f} "
        f"order_acc={bm['order_acc']:.4f} are vs {d.aromatic_mode} labels, "
        f"so ignore them if mode=kekule")
    del base
    if str(DEVICE).startswith("cuda"):
        torch.cuda.empty_cache()

In [ ]:
# ------------------------------------------ train ------------------------------------------
opt = AdamW(model.parameters(), lr=LR, weight_decay=1e-8)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

CFG = {k: str(globals()[k]) for k in
       ("N_HIDDEN", "EPOCHS", "BATCH", "LR", "LOSS", "CANONICAL", "WARM_START",
        "HEAD_SCALE", "WEIGHT_CAP", "SELECT", "SEED", "DATA_DIR", "OUT_DIR")}

best, no_imp, total_time = -1.0, 0, 0.0
for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    ep = train_idx[torch.randperm(train_idx.shape[0])]
    run, run_wrong, ns = 0.0, 0.0, 0
    pbar = tqdm(range(0, ep.shape[0] - BATCH + 1, BATCH), desc=f"h{N_HIDDEN} ep{epoch}")
    for s in pbar:
        bidx = ep[s : s + BATCH]
        el, dist, adj_in, tgt, na = build_inputs(d, bidx, DEVICE)
        logits = model(elements=el, dist_mat=dist, adj_mat=adj_in)
        loss = compute_loss(logits, tgt, na, LOSS, w)
        if not torch.isfinite(loss):
            log(f"WARN non-finite loss ep={epoch}")
            continue
        if ns == 0 and epoch == 0:
            log(f"first-batch loss (pre-update) = {loss.item():.4f}")
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        run += loss.item()
        run_wrong += wrong_bonds(logits.detach(), tgt, na)["wrong"].mean().item()
        ns += 1
        if ns % 50 == 0:
            pbar.set_postfix(loss=f"{run / ns:.4f}", wrong=f"{run_wrong / ns:.4f}")
    sched.step()

    mt = evaluate(model, d, val_idx, DEVICE, BATCH, w, N_VALIDITY, LOSS)
    total_time += time.time() - epoch_start
    log(f"ep={epoch} train_loss={run / max(ns, 1):.4f} train_wrong={run_wrong / max(ns, 1):.4f} "
        f"val_loss={mt['loss']:.4f} val_wrong={mt['wrong']:.4f} "
        f"valid={mt['valid']:.3f} exact={mt['exact']:.3f} "
        f"bond_f1={mt['bond_f1']:.4f} bond_p={mt['bond_p']:.4f} bond_r={mt['bond_r']:.4f} "
        f"order_acc={mt['order_acc']:.4f} avg_epoch={total_time / (epoch + 1):.0f}s")

    ckpt = {
        "state_dict": model.state_dict(),
        "n_hidden": N_HIDDEN,
        "epoch": epoch,
        "metrics": mt,
        "aromatic_mode": d.aromatic_mode,
        "args": CFG,
    }
    torch.save(ckpt, OUT_DIR / f"latest_seer_{N_HIDDEN}.pt")
    score = mt["valid"] if SELECT == "valid" else -mt["wrong"]
    if score > best:
        best, no_imp = score, 0
        torch.save(ckpt, OUT_DIR / f"best_seer_{N_HIDDEN}.pt")
        log(f"ckpt best {SELECT}={abs(score):.4f}")
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            log(f"early stop ep={epoch}")
            break

log(f"done n_hidden={N_HIDDEN} best_{SELECT}={abs(best):.4f}")